# CanAI Café — Forecasting Model
**Owner:** Member C  
**Input:** `../data/clean/data_clean.csv`  
**Output:** `../outputs/forecast_results.csv`

**Structure:**
1. Data Preparation — load, aggregate, configure shared TimeSeriesSplit  
2. Series Visualisation  
3. Prophet — evaluate across folds, store metrics  
4. SARIMA — evaluate across folds, store metrics  
5. XGBoost — evaluate across folds, store metrics  
6. Model Comparison — side-by-side metrics table, select winner  
7. Final Forecast & Export — refit winner on full data, save `forecast_results.csv`

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../models'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
import mlflow_setup
mlflow_setup.setup()
print(f"Tracking URI : {mlflow_setup.TRACKING_URI}")
print(f"Experiment   : {mlflow_setup.EXPERIMENT_NAME}")

## MLflow Tracking

All three model modules log to the shared experiment **`canai-cafe-forecasting`** stored under `mlruns/` at the project root.

To open the tracking UI, run this from the project root:
```
mlflow ui --backend-store-uri ./mlruns --port 5000
```
Then open **http://localhost:5000**

## 1. Data Preparation

In [ ]:
df = pd.read_csv('../data/clean/data_clean.csv', parse_dates=['Transaction Date'])

# Aggregate to daily total revenue — shared input for all three models
daily_revenue = (
    df.groupby('Transaction Date')['Total Spent']
    .sum()
    .sort_index()
    .rename('revenue')
)

print(f"Series: {len(daily_revenue)} days")
print(f"Range:  {daily_revenue.index.min().date()} to {daily_revenue.index.max().date()}")
print(f"Daily mean: ${daily_revenue.mean():.2f} | std: ${daily_revenue.std():.2f}")

# Walk-forward cross-validation — identical across all three models (see docs/model_selection_rationale.md §6.3)
tscv = TimeSeriesSplit(n_splits=5, test_size=30)
print(f"\nTimeSeriesSplit: {tscv.n_splits} folds, 30-day test windows")

## 2. Series Visualisation

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(daily_revenue.index, daily_revenue.values)
plt.title('Daily Revenue — CanAI Café 2023')
plt.xlabel('Date')
plt.ylabel('Revenue ($)')
plt.tight_layout()
plt.show()

## 3. Prophet

In [ ]:
import prophet_model

prophet_metrics = prophet_model.evaluate(daily_revenue, tscv)
prophet_metrics

## 4. SARIMA

In [ ]:
import sarima_model

sarima_metrics = sarima_model.evaluate(daily_revenue, tscv)
sarima_metrics

## 5. XGBoost

In [ ]:
import xgboost_model

xgboost_metrics = xgboost_model.evaluate(daily_revenue, tscv)
xgboost_metrics

## 6. Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model':     ['Prophet', 'SARIMA', 'XGBoost'],
    'Mean MAE':  [prophet_metrics['mean_mae'],  sarima_metrics['mean_mae'],  xgboost_metrics['mean_mae']],
    'Mean RMSE': [prophet_metrics['mean_rmse'], sarima_metrics['mean_rmse'], xgboost_metrics['mean_rmse']],
    'Mean MAPE': [prophet_metrics['mean_mape'], sarima_metrics['mean_mape'], xgboost_metrics['mean_mape']],
})
results = results.set_index('Model').round(2)

# Flag the winner (lowest MAPE)
winner = results['Mean MAPE'].idxmin()
print(f"Selected model: {winner}")
results

## 7. Final Forecast & Export

Refit the winning model on the full 365-day series, generate a 6-month forward forecast, and save `forecast_results.csv`.

In [ ]:
model_map = {
    'Prophet': prophet_model,
    'SARIMA':  sarima_model,
    'XGBoost': xgboost_model,
}

forecast_6m = model_map[winner].forecast(daily_revenue, periods=180)

# Aggregate daily forecast to monthly totals
forecast_6m.to_csv('../outputs/forecast_results.csv', index=False)
print('Saved: ../outputs/forecast_results.csv')
forecast_6m